# Importância de features em cenários bancários sintéticos: sem Boruta

Este notebook mede a importância nativa das features antes de qualquer seleção Boruta. Compara BankaiRF instalado do local build com `extra_trees=False` e `extra_trees=True`, Random Forest do scikit-learn, Extra Trees, LightGBM RF com `extra_trees=True` e XGBoost RF nos mesmos quatro cenários sintéticos.

O ajuste usa 80% das linhas e reporta ROC-AUC e average precision em um holdout estratificado de 20%. A qualidade do ranking é comparada com a verdade conhecida por construção: average precision do ranking informativo e precision/recall entre as top 24 features.

As variantes Bankai usam os mesmos dados, parâmetros e seed do modelo em cada repetição; a ordem entre elas alterna para reduzir viés de tempo. O notebook registra tempo de treino e métricas de qualidade. O modo light fica ativado por padrão; para avaliação completa, altere `LIGHT_MODE` para `False`: 500.000 linhas, 500 features e 10 medições por modelo/cenário, mais aquecimentos. O Bankai usa aceleração por histograma com `max_bins=63`; Extra Trees amostra um limiar por feature amostrada.

## 1. Instalar dependências

In [ ]:
import subprocess, sys
from pathlib import Path

repo = Path.cwd().resolve()
repo = repo.parent if repo.name == 'benchmarks' else repo
subprocess.check_call(['uv', 'pip', 'install', '--python', sys.executable, '--no-deps', '-e', str(repo)])
subprocess.check_call(['uv', 'pip', 'install', '--python', sys.executable, 'scikit-learn==1.9.1', 'lightgbm', 'xgboost', 'psutil', 'threadpoolctl', 'pandas', 'numpy', 'scipy'])


Reinicie o kernel se solicitado. As versões fixas ajudam outras pessoas a reproduzir o ambiente.

## 2. Recursos disponíveis

In [ ]:
import os, platform, sys
import psutil
from threadpoolctl import threadpool_info

CPU_COUNT = os.cpu_count() or 1
physical = psutil.cpu_count(logical=False) or CPU_COUNT
memory = psutil.virtual_memory()
print({
    'machine': platform.machine(), 'platform': platform.platform(),
    'python': sys.version, 'logical_cpus': CPU_COUNT,
    'physical_cores': physical, 'memory_total_gib': round(memory.total / 2**30, 2),
    'memory_available_gib': round(memory.available / 2**30, 2),
    'threadpools_before_import': threadpool_info(),
})
N_JOBS = 6  # limite para benchmarks locais

## 3. Configuração

In [ ]:
from pathlib import Path

LIGHT_MODE = False  # mude para False para o benchmark completo
BASE_SEED = 20261012
N_WARMUP = 1
WORKDIR = Path.cwd().resolve()
REPO = WORKDIR.parent if WORKDIR.name == 'benchmarks' else WORKDIR
from datetime import datetime, timezone
RESULT_DIR = REPO / 'benchmarks' / ('campaign-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = RESULT_DIR / ('raw-light.csv' if LIGHT_MODE else 'raw-full.csv')
IMPORTANCE_PATH = RESULT_DIR / ('importance-light.csv' if LIGHT_MODE else 'importance-full.csv')

MAX_BINS = 63

if LIGHT_MODE:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 20_000, 100, 24
    N_REPEATS, N_TREES = 1, 20
else:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 500_000, 500, 24
    N_REPEATS, N_TREES = 3, 100

print({'light_mode': LIGHT_MODE, 'rows_per_scenario': N_ROWS,
       'features': N_FEATURES, 'informative_features': N_INFORMATIVE,
       'measured_repeats_per_model_scenario': N_REPEATS,
       'warmups_per_model_scenario': N_WARMUP, 'trees': N_TREES,
       'test_fraction': 0.20, 'n_jobs': N_JOBS,
       'result_file': str(RAW_PATH)})

### Recursos e duração

Modo completo: 4 cenários × 6 estimadores × 10 medições + 24 aquecimentos = 264 ajustes. A matriz float32 de 500.000 × 500 usa cerca de 0,93 GiB; treino e holdout geram cópias adicionais. Este notebook não cria a matriz shadow do Boruta. Confira a memória disponível.

Todos os estimadores usam todos os CPUs lógicos detectados. Cada repetição compartilha a mesma seed de dados e split entre modelos. As duas variantes Bankai também compartilham a seed do modelo, com a ordem alternada entre repetições.

## 4. Cenários e premissas

1. **Inadimplência de crédito (1,5% positivos):** prever atraso grave futuro; sinais de comprometimento de renda, uso de crédito, histórico, emprego e idade de crédito. Sinais fracos e correlações motivam árvores moderadas e folhas mínimas maiores.
2. **Fraude em transações (0,2% positivos):** valor atípico, horário/dispositivo incomum, velocidade e interações. Evento muito raro motiva árvores um pouco mais profundas para capturar combinações, mantendo folhas mínimas para limitar sobreajuste.
3. **Churn (6% positivos):** queda de atividade e saldo, reclamações, tarifas e recência de contato. Relações por segmento justificam profundidade intermediária.
4. **Aceitação de oferta (18% positivos):** capacidade financeira, engajamento, adequação e canal. Mais positivos e sinais relativamente diretos permitem árvores mais rasas.

Cada cenário tem 24 features informativas nos índices 0–23 e ruído nas restantes. Há proxies correlacionadas, limiares, curvas e interações. Nenhuma coluna revela o rótulo diretamente. São hipóteses sintéticas controladas, não retratos de clientes reais.

As escolhas iniciais de regularização estão na tabela SCENARIOS: folhas maiores e profundidade moderada para crédito raro; mais profundidade com folhas mínimas para fraude; valores intermediários para churn; árvores rasas para aceitação. São premissas de partida, não resultados de busca de hiperparâmetros.

Todos os modelos de floresta usam o mesmo número de árvores (20 no modo light; 100 no completo), 80% de amostras por árvore, amostragem de features próxima à raiz de p, profundidade e folha derivados da mesma tabela e pesos de classe balanceados. As variantes Bankai diferem somente em `extra_trees`; ambas usam `extra_seed=6`. min_child_weight do XGBoost é massa Hessiana e não equivale exatamente à contagem min_samples_leaf; usamos leaf × 0,25 como aproximação e reportamos essa diferença. Bankai usa max_bins=63; LightGBM também ativa `extra_trees=True` e ambos LightGBM e XGBoost usam histogramas com 63 bins.

| Cenário | max_depth | min_samples_leaf | amostra de linhas |
|---|---:|---:|---:|
| Inadimplência | 8 | 500 | 80% |
| Fraude | 9 | 750 | 80% |
| Churn | 7 | 250 | 80% |
| Aceitação | 6 | 300 | 80% |

A folha mínima maior nos cenários com menos positivos reduz folhas sustentadas por poucos eventos. Todos os braços mantêm 100 árvores e feature sampling próximo de raiz de p.

## 5. Importações e confirmação do Bankai do local build

In [ ]:
import gc, json, math, os, time, warnings
from importlib import metadata
from pathlib import Path

repo = Path.cwd().resolve()
repo = repo.parent if repo.name == 'benchmarks' else repo
python_source = (repo / 'python').resolve()
import sys
sys.path.insert(0, str(python_source))

import numpy as np
import pandas as pd
from scipy.special import expit
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from threadpoolctl import threadpool_limits
import bankai_random_forest
from bankai_random_forest import BankaiRandomForestClassifier

package_file = Path(bankai_random_forest.__file__).resolve()
extension_file = Path(bankai_random_forest._core.__file__).resolve()
dist = metadata.distribution('bankai-random-forest')
EXPECTED_BANKAI_VERSION = '0.1.0a12'
if dist.version != EXPECTED_BANKAI_VERSION:
    raise RuntimeError(f'Esperado Bankai {EXPECTED_BANKAI_VERSION}, encontrado {dist.version}')
direct_url = dist.read_text('direct_url.json')
if not package_file.is_relative_to(python_source):
    raise RuntimeError(f'Bankai não foi importado do checkout local: {package_file}')
if direct_url:
    install_origin = json.loads(direct_url)
    if not install_origin.get('dir_info', {}).get('editable', False):
        raise RuntimeError('Bankai não está instalado em modo editável a partir do checkout local.')
def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        # Some notebook environments expose an importable module without its
        # wheel metadata. Keep the environment report useful in that case.
        return 'metadata unavailable'

versions = {
    'bankai': dist.version,
    'bankai_package': str(package_file),
    'bankai_native_extension': str(extension_file),
    'scikit_learn': installed_version('scikit-learn'),
    'lightgbm': installed_version('lightgbm'),
    'xgboost': installed_version('xgboost'),
}
if 'boruta' in globals():
    versions['boruta'] = installed_version('Boruta')
print(versions)

## 6. Geradores dos datasets

As features 0–23 são informativas por construção; a tabela `SCENARIOS` registra prevalência e parâmetros do estimador assumidos para cada problema. Todas as bibliotecas recebem a mesma seed de dados em cada repetição e os mesmos limites principais. As variantes Bankai têm também a mesma seed de modelo em cada par; apenas `extra_trees` muda (e ambas usam `extra_seed=6`). Os demais modelos recebem seeds determinísticas próprias. São usados limites compatíveis para profundidade, tamanho mínimo de folha, número de árvores, amostragem de linhas e amostragem de features. Diferenças internas entre os algoritmos permanecem e são descritas nos resultados.

In [ ]:
SCENARIOS = {
    'credit_default': {'prevalence': 0.015, 'description': 'Inadimplência futura', 'max_depth': 8, 'min_leaf': 500, 'row_fraction': 0.80},
    'transaction_fraud': {'prevalence': 0.002, 'description': 'Fraude confirmada', 'max_depth': 9, 'min_leaf': 750, 'row_fraction': 0.80},
    'customer_churn': {'prevalence': 0.060, 'description': 'Encerramento futuro', 'max_depth': 7, 'min_leaf': 250, 'row_fraction': 0.80},
    'offer_acceptance': {'prevalence': 0.180, 'description': 'Aceitação de oferta', 'max_depth': 6, 'min_leaf': 300, 'row_fraction': 0.80},
}

def calibrate_intercept(signal, prevalence):
    signal = (signal - signal.mean()) / max(signal.std(), 1e-8)
    lo, hi = -30.0, 30.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if expit(signal + mid).mean() > prevalence:
            hi = mid
        else:
            lo = mid
    return signal + (lo + hi) / 2

def make_dataset(scenario, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(N_ROWS, N_FEATURES)).astype(np.float32)
    # Indicadores e variáveis monetárias com assimetria, comuns em dados bancários.
    X[:, 4:8] = rng.binomial(1, [0.12, 0.08, 0.18, 0.35], size=(N_ROWS, 4)).astype(np.float32)
    X[:, 8:12] = np.log1p(rng.lognormal(mean=2.0, sigma=0.8, size=(N_ROWS, 4))).astype(np.float32)
    X[:, 8:12] = (X[:, 8:12] - X[:, 8:12].mean(axis=0)) / (X[:, 8:12].std(axis=0) + 1e-6)
    # Proxies correlacionadas; todas continuam informativas para a verdade sintética.
    X[:, 16:20] = 0.75 * X[:, 0:4] + 0.66 * rng.normal(size=(N_ROWS, 4)).astype(np.float32)
    z = X

    if scenario == 'credit_default':
        signal = (1.0*z[:,0] + .85*z[:,1] + .65*z[:,2] + .5*z[:,3]
                  + .8*(z[:,4] > .5) + .7*(z[:,5] > .5) + .5*z[:,6] - .4*z[:,7]
                  + .65*z[:,8] - .55*z[:,9] + .45*z[:,10] + .35*z[:,11]
                  + .7*np.maximum(z[:,12]-.8, 0) + .6*(z[:,13]**2-1)
                  + .65*z[:,14]*z[:,15] + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'transaction_fraud':
        signal = (1.0*z[:,0] + .8*np.maximum(z[:,1]-1, 0) + .7*(z[:,2] > 1.2)
                  + .9*z[:,3]*z[:,4] + .7*(z[:,5] > .5) + .5*z[:,6]
                  + .8*np.abs(z[:,7]) + .55*z[:,8] + .5*z[:,9]
                  + .65*(z[:,10]**2-1) + .7*z[:,11]*z[:,12]
                  + .5*z[:,13] + .4*z[:,14] + .35*z[:,15]
                  + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'customer_churn':
        signal = (-.9*z[:,0] - .8*z[:,1] + .7*z[:,2] + .6*z[:,3]
                  + .65*(z[:,4] > .5) + .7*(z[:,5] > .5) + .5*z[:,6]
                  + .45*z[:,7] - .65*z[:,8] + .55*z[:,9] + .45*z[:,10]
                  + .6*np.maximum(z[:,11]-.7, 0) + .65*z[:,12]*z[:,13]
                  + .5*np.tanh(z[:,14]) + .4*z[:,15] + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'offer_acceptance':
        signal = (.8*z[:,0] + .7*z[:,1] - .5*z[:,2] + .65*z[:,3]
                  + .6*(z[:,4] > .5) + .45*(z[:,5] > .5) + .5*z[:,6]
                  + .4*z[:,7] + .75*z[:,8] + .55*z[:,9] - .4*z[:,10]
                  + .6*(z[:,11]**2-1) + .65*z[:,12]*z[:,13]
                  + .5*np.tanh(z[:,14]) + .35*z[:,15] + .18*z[:,16:24].sum(axis=1))
    else:
        raise ValueError(f'Cenário desconhecido: {scenario}')
    logits = calibrate_intercept(signal, SCENARIOS[scenario]['prevalence'])
    y = rng.binomial(1, expit(logits)).astype(np.int8)
    return X, y, np.arange(N_FEATURES) < N_INFORMATIVE

print({name: info for name, info in SCENARIOS.items()})

## 7. Modelos e cálculo de importâncias

In [ ]:
class XGBoostForestEstimator(XGBClassifier):
    # Expose one sklearn n_estimators value to Boruta while fitting that many
    # parallel trees in a single XGBoost boosting round.
    def fit(self, X, y, **kwargs):
        tree_count = self.n_estimators
        self.set_params(n_estimators=1, num_parallel_tree=tree_count)
        return super().fit(X, y, **kwargs)

def estimator_factory(model_name, seed, positive_weight, scenario):
    cfg = SCENARIOS[scenario]
    depth, leaf, row_fraction = cfg['max_depth'], cfg['min_leaf'], cfg['row_fraction']
    p_augmented = N_FEATURES
    node_fraction = min(1.0, math.sqrt(p_augmented) / p_augmented)
    common_seed = int(seed % (2**31 - 1))
    if model_name in ('BankaiRF (local build, extra_trees=False)', 'BankaiRF (local build, extra_trees=True)'):
        return BankaiRandomForestClassifier(
            n_estimators=N_TREES, max_depth=depth, min_samples_leaf=leaf, max_features='sqrt',
            max_samples=None, bootstrap=False, max_bins=(None if model_name.endswith('extra_trees=True)') else MAX_BINS),
            importance_type='gain', class_weight='balanced',
            extra_trees=model_name.endswith('extra_trees=True)'), extra_seed=6,
            random_state=common_seed, n_jobs=N_JOBS)
    if model_name == 'Random Forest (sklearn)':
        return RandomForestClassifier(
            n_estimators=N_TREES, max_depth=depth, min_samples_leaf=leaf, max_features='sqrt',
            max_samples=None, bootstrap=False, class_weight='balanced',
            random_state=common_seed, n_jobs=N_JOBS)
    if model_name == 'Extra Trees':
        return ExtraTreesClassifier(
            n_estimators=N_TREES, max_depth=depth, min_samples_leaf=leaf, max_features='sqrt',
            bootstrap=False, max_samples=None, class_weight='balanced',
            random_state=common_seed, n_jobs=N_JOBS)
    if model_name == 'LightGBM RF (extra_trees=True)':
        return LGBMClassifier(
            boosting_type='rf', objective='binary', n_estimators=N_TREES,
            max_depth=depth, num_leaves=min(2**depth, 256), min_child_samples=leaf, max_bin=MAX_BINS,
            extra_trees=True,
            bagging_fraction=row_fraction, bagging_freq=1,
            feature_fraction=1.0, feature_fraction_bynode=node_fraction,
            scale_pos_weight=positive_weight, importance_type='gain',
            random_state=common_seed, n_jobs=N_JOBS, verbosity=-1)
    if model_name == 'XGBoost RF':
        return XGBoostForestEstimator(
            objective='binary:logistic', n_estimators=N_TREES,
            num_parallel_tree=N_TREES, max_depth=depth, min_child_weight=leaf * 0.25, tree_method='hist',
            max_bin=MAX_BINS, subsample=row_fraction,
            colsample_bytree=1.0, colsample_bynode=node_fraction,
            learning_rate=1.0, scale_pos_weight=positive_weight,
            importance_type='gain', random_state=common_seed,
            n_jobs=N_JOBS, verbosity=0)
    raise ValueError(model_name)

BANKAI_OFF = 'BankaiRF (local build, extra_trees=False)'
BANKAI_ON = 'BankaiRF (local build, extra_trees=True)'
MODEL_NAMES = [BANKAI_OFF, BANKAI_ON, 'Random Forest (sklearn)', 'Extra Trees', 'LightGBM RF (extra_trees=True)', 'XGBoost RF']

## 8. Aquecimento, holdout, medições e retomada

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
TOP_K = N_INFORMATIVE

def evaluate_one(X, y, informative, scenario, model_name, repeat_no, data_seed, model_seed):
    indices = np.arange(len(y))
    tr, te = train_test_split(indices, test_size=0.20, stratify=y,
                              random_state=int((data_seed+73) % (2**31-1)))
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
    pos_weight = float((ytr==0).sum()/max((ytr==1).sum(),1))
    model = estimator_factory(model_name, model_seed, pos_weight, scenario)
    t=time.perf_counter()
    with threadpool_limits(limits=N_JOBS): model.fit(Xtr,ytr)
    fit_s=time.perf_counter()-t
    t=time.perf_counter()
    with threadpool_limits(limits=N_JOBS): prob=model.predict_proba(Xte)[:,1]
    predict_s=time.perf_counter()-t
    imp=np.nan_to_num(np.asarray(model.feature_importances_,dtype=np.float64),nan=0.0,posinf=0.0,neginf=0.0)
    ranking=np.argsort(-imp,kind='stable'); rank_position=np.empty_like(ranking)
    rank_position[ranking]=np.arange(1,len(ranking)+1); top=ranking[:TOP_K]
    tp=int(informative[top].sum())
    row={'scenario':scenario,'model':model_name,'repeat':repeat_no,'data_seed':data_seed,
         'model_seed':model_seed,'fit_seconds':fit_s,'prediction_seconds':predict_s,
         'feature_average_precision':float(average_precision_score(informative.astype(int),imp)),
         'recall_at_k':tp/int(informative.sum()),'precision_at_k':tp/max(len(top),1),'top_k':TOP_K,
         'roc_auc':float(roc_auc_score(yte,prob)),'pr_auc':float(average_precision_score(yte,prob)),
         'positive_prevalence':float(y.mean()),'train_rows':len(tr),'test_rows':len(te),
         'features':X.shape[1],'informative_features':int(informative.sum()),'n_jobs':N_JOBS,
         'trees':N_TREES,'max_depth':SCENARIOS[scenario]['max_depth'],
         'min_samples_leaf':SCENARIOS[scenario]['min_leaf'],'row_fraction':(1.0 if model_name in (BANKAI_OFF,BANKAI_ON,'Random Forest (sklearn)','Extra Trees') else SCENARIOS[scenario]['row_fraction']),
         'bankai_max_bins':MAX_BINS,'bankai_extra_trees':(model_name.endswith('extra_trees=True') if model_name.startswith('BankaiRF') else None),
         'bankai_extra_seed':(6 if model_name.startswith('BankaiRF') else None),'run_signature':RUN_SIGNATURE}
    scores=[{'scenario':scenario,'model':model_name,'repeat':repeat_no,'data_seed':data_seed,
             'feature_index':int(j),'importance':float(imp[j]),
             'rank':int(rank_position[j]),'is_informative':bool(informative[j]),
             'run_signature':RUN_SIGNATURE} for j in range(len(imp))]
    del model,Xtr,Xte,ytr,yte,prob
    return row,scores

RUN_SIGNATURE=json.dumps({'light_mode':LIGHT_MODE,'rows':N_ROWS,'features':N_FEATURES,
 'repeats':N_REPEATS,'trees':N_TREES,'base_seed':BASE_SEED,'n_jobs':N_JOBS,
 'max_bins':MAX_BINS,'bankai_extra_trees':[False,True],'bankai_extra_seed':6,
 'scenarios':SCENARIOS,'models':MODEL_NAMES,'top_k':TOP_K,
 'implementation': __import__('hashlib').sha256(extension_file.read_bytes()).hexdigest(),
 'sources': {str(p.relative_to(REPO)): __import__('hashlib').sha256(p.read_bytes()).hexdigest() for folder in ('src', 'crates/xrf/src', 'python/bankai_random_forest') for p in (REPO/folder).rglob('*') if p.suffix in ('.rs', '.py')}},sort_keys=True)
(RESULT_DIR/'manifest.json').write_text(json.dumps({'signature':json.loads(RUN_SIGNATURE), 'versions':versions, 'hardware':{'platform':platform.platform(), 'logical_cpus':CPU_COUNT, 'physical_cores':physical, 'memory_bytes':memory.total}},indent=2),encoding='utf-8')
records=pd.read_csv(RAW_PATH).to_dict('records') if RAW_PATH.exists() else []
score_records=pd.read_csv(IMPORTANCE_PATH).to_dict('records') if IMPORTANCE_PATH.exists() else []
if any(r.get('run_signature')!=RUN_SIGNATURE for r in records+score_records):
    raise RuntimeError('CSVs existentes usam outra configuração; use uma pasta limpa.')
done={(r['scenario'],r['model'],int(r['repeat'])) for r in records if int(r['repeat'])>=0}

def save_checkpoint(frame, path):
    tmp=path.with_name(path.name + '.tmp')
    frame.to_csv(tmp,index=False)
    tmp.replace(path)

for scenario in ['transaction_fraud'] + [s for s in SCENARIOS if s != 'transaction_fraud']:
    si=list(SCENARIOS).index(scenario)
    print(); print(f'=== {scenario} ===',flush=True)
    warm_seed=BASE_SEED+si*1_000_000+777
    Xw,yw,_=make_dataset(scenario,warm_seed)
    tr,_=train_test_split(np.arange(len(yw)),test_size=.2,stratify=yw,random_state=(warm_seed+73)%(2**31-1))
    pos=float((yw[tr]==0).sum()/max((yw[tr]==1).sum(),1))
    for mi,name in enumerate(MODEL_NAMES):
        warm_model_seed=BASE_SEED+si*1_000_000+800+(0 if name.startswith('BankaiRF') else mi*101)
        model=estimator_factory(name,warm_model_seed,pos,scenario)
        with threadpool_limits(limits=N_JOBS): model.fit(Xw[tr],yw[tr])
        del model; gc.collect(); print('warmup complete:',name,flush=True)
    del Xw,yw,tr; gc.collect()
    for rep in range(N_REPEATS):
        if all((scenario,name,rep) in done for name in MODEL_NAMES): continue
        data_seed=BASE_SEED+si*1_000_000+rep*10_000+500
        X,y,informative=make_dataset(scenario,data_seed)
        # Alternate the paired Bankai variants' position; keep each model's seed stable.
        ordered_names=MODEL_NAMES.copy()
        if rep % 2: ordered_names.reverse()
        for name in ordered_names:
            mi=MODEL_NAMES.index(name)
            if (scenario,name,rep) in done: continue
            model_offset=0
            model_seed=BASE_SEED+si*1_000_000+rep*10_000+model_offset+900
            print(f'{scenario} | {rep+1}/{N_REPEATS} | {name}',flush=True)
            row,scores=evaluate_one(X,y,informative,scenario,name,rep,data_seed,model_seed)
            records.append(row); score_records.extend(scores)
            score_frame=(pd.DataFrame(score_records)
                .drop_duplicates(['scenario','model','repeat','feature_index'],keep='last'))
            save_checkpoint(score_frame,IMPORTANCE_PATH)
            save_checkpoint(pd.DataFrame(records),RAW_PATH)
            done.add((scenario,name,rep))
            print({k:row[k] for k in ('fit_seconds','feature_average_precision','recall_at_k','precision_at_k','roc_auc','pr_auc')},flush=True)
        del X,y; gc.collect()
print('Metrics:',RAW_PATH); print('Per-feature importance:',IMPORTANCE_PATH)

## 9. Resumo e melhor ranking por cenário

In [ ]:
raw=pd.read_csv(RAW_PATH)
summary=(raw.groupby(['scenario','model'],sort=False)
 .agg(runs=('repeat','nunique'),n_jobs=('n_jobs','first'),bankai_max_bins=('bankai_max_bins','first'),max_depth=('max_depth','first'),
      min_samples_leaf=('min_samples_leaf','first'),row_fraction=('row_fraction','first'),
      fit_median_s=('fit_seconds','median'),fit_iqr_s=('fit_seconds',lambda x:x.quantile(.75)-x.quantile(.25)),
      feature_ap_median=('feature_average_precision','median'),
      recall_at_k_median=('recall_at_k','median'),precision_at_k_median=('precision_at_k','median'),
      roc_auc_median=('roc_auc','median'),pr_auc_median=('pr_auc','median')).reset_index())
summary['importance_rank']=summary.groupby('scenario')['feature_ap_median'].rank(method='min',ascending=False)
summary=summary.sort_values(['scenario','importance_rank','fit_median_s'])
summary.to_csv(RESULT_DIR/('summary-light.csv' if LIGHT_MODE else 'summary-full.csv'),index=False)
display(summary)

# Primary paired comparison: Bankai Extra Trees versus sklearn Extra Trees, with identical nominal seeds.
REFERENCE_MODEL = 'Extra Trees'
bankai_pair=(raw[raw['model'].isin([REFERENCE_MODEL,BANKAI_ON])]
 .pivot(index=['scenario','repeat','data_seed','model_seed'],columns='model',
        values=['fit_seconds','feature_average_precision','recall_at_k','precision_at_k','roc_auc','pr_auc'])
 .dropna().reset_index())
bankai_pair.columns=['_'.join(str(part) for part in col if str(part)) if isinstance(col,tuple) else col
                     for col in bankai_pair.columns]
bankai_pair['fit_time_ratio_on_over_off']=bankai_pair[f'fit_seconds_{BANKAI_ON}']/bankai_pair[f'fit_seconds_{REFERENCE_MODEL}']
bankai_pair['fit_time_delta_on_minus_off_s']=bankai_pair[f'fit_seconds_{BANKAI_ON}']-bankai_pair[f'fit_seconds_{REFERENCE_MODEL}']
bankai_pair['recall_at_k_delta_on_minus_off']=bankai_pair[f'recall_at_k_{BANKAI_ON}']-bankai_pair[f'recall_at_k_{REFERENCE_MODEL}']
bankai_pair['false_discoveries_delta_on_minus_off']=(TOP_K*(1-bankai_pair[f'precision_at_k_{BANKAI_ON}']))-(TOP_K*(1-bankai_pair[f'precision_at_k_{REFERENCE_MODEL}']))
# Compute paired deltas directly for ranking AP, avoiding comparisons of unpaired medians.
bankai_pair['feature_ap_delta_on_minus_off']=bankai_pair[f'feature_average_precision_{BANKAI_ON}']-bankai_pair[f'feature_average_precision_{REFERENCE_MODEL}']
pair_summary=bankai_pair.groupby('scenario',sort=False).agg(
    paired_runs=('repeat','nunique'),fit_time_ratio_median=('fit_time_ratio_on_over_off','median'),
    fit_time_delta_median_s=('fit_time_delta_on_minus_off_s','median'),
    feature_ap_delta_median=('feature_ap_delta_on_minus_off','median'),
    recall_at_k_delta_median=('recall_at_k_delta_on_minus_off','median'),
    false_discoveries_delta_median=('false_discoveries_delta_on_minus_off','median')).reset_index()
bankai_pair.to_csv(RESULT_DIR/('bankai-paired-light.csv' if LIGHT_MODE else 'bankai-paired-full.csv'),index=False)
pair_summary.to_csv(RESULT_DIR/('bankai-paired-summary-light.csv' if LIGHT_MODE else 'bankai-paired-summary-full.csv'),index=False)
print('Comparação pareada Bankai (extra_trees=True menos False; razão de tempo <1 favorece True):')
display(pair_summary)
best=summary.drop_duplicates('scenario',keep='first')
print('Melhor ranking nativo por cenário: maior AP das features; desempate por runtime.')
display(best[['scenario','model','feature_ap_median','recall_at_k_median','precision_at_k_median',
              'roc_auc_median','pr_auc_median','fit_median_s','runs']])
print('Resultados em:',RESULT_DIR)

## 10. Interpretação

Feature average precision mede o ranking inteiro contra o rótulo conhecido de feature informativa. Recall@24 e precision@24 mostram quantas das 24 informativas aparecem nas 24 primeiras. ROC-AUC e PR-AUC são medidas preditivas em holdout e apenas contextualizam a importância.

As escalas nativas de importance diferem entre bibliotecas; compare ranking e métricas, não os valores brutos. A tabela pareada compara tempo de treino e diferenças de qualidade entre `extra_trees=True` e `False`; razão de tempo abaixo de 1 significa que o modo ligado foi mais rápido. O CSV detalhado guarda score e posição de cada feature por repetição para avaliar estabilidade. O notebook companheiro mede quanto Boruta muda essa qualidade ao comparar features com sombras.

Para uso bancário real, valide em períodos futuros, investigue leakage e estabilidade por segmentos. Importância preditiva não demonstra causalidade.